# Per-Pulse Stable Merge
Scan a data folder for Ramsey and readout-power subfolders.  
1. Use `charge_gate_ramsey` runs to identify **stable charge intervals**.  
2. Parse `charge_gate_readout_power_with_ref` folder names to extract the **pulse name** and repeat index.  
3. For each pulse name, keep only runs within stable intervals, merge across repeats, and save to a separate HDF5 file.

In [2]:
import gc
import json
import os
import re
from pathlib import Path
from collections import defaultdict

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from scqat.protocols.charge_gate_ramsey import ChargeGateRamseyAnalyzer
from scqat.parsers.xarray_h5_parser import load_xarray_h5
from scqat.parsers.qualibrate_parser import repetition_data, parse_timestamp

## 1. Configuration

In [3]:
# ---- INPUT ----
DATA_ROOT = r"D:\SynologyDrive\LiChiehHsiao\AS\SynologyDrive\data\MIST\20260402\Q1\ng01_ref0165_test04\MWF\op_A13_ngv08_5T"

# ---- Ramsey settings ----
RAMSEY_FOLDER_PATTERN = "*charge_gate_ramsey*"
RAMSEY_SIGNAL_VAR = "I"  # variable name to rename to "signal"

ANALYSIS_KWARGS = {
    "f_c_fixed": 0.0002,
    "abscos_frequency_fixed": 0.59,
    "abscos_phase_bounds": (-0.5, 0.5),
}

# ---- Stability threshold ----
PHASE_THRESHOLD = 0.01  # max |Δphase| between consecutive Ramsey runs

# ---- Readout-power settings ----
RO_FOLDER_PATTERN = "*charge_gate_readout_power_with_ref*"
RO_SIGNAL_VARS = ["I1_1", "I1_2", "I1_3", "Q1_1", "Q1_2", "Q1_3"]

# Regex to extract repeat index and pulse name from the folder name.
# Example: "#851_LCH_charge_gate_readout_power_with_ref_0_readout_three_step_184606"
#   -> repeat_index = 0, pulse_name = "readout_three_step"
# Example: "#852_LCH_charge_gate_readout_power_with_ref_0_readout_three_step_100_190012"
#   -> repeat_index = 0, pulse_name = "readout_three_step_100"
# Example: "#853_LCH_charge_gate_readout_power_with_ref_0_square_190506"
#   -> repeat_index = 0, pulse_name = "square"
RO_NAME_REGEX = r"charge_gate_readout_power_with_ref_(\d+)_(.+?)_\d{6}$"

# ---- Output ----
OUTPUT_DIR = str(Path(DATA_ROOT) / "per_pulse")
FIGURE_SAVE_PATH = os.path.join(DATA_ROOT, "figures", "per_pulse")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FIGURE_SAVE_PATH, exist_ok=True)

## 2. Ramsey Stability Analysis
Run `ChargeGateRamseyAnalyzer` on every Ramsey folder and identify stable time intervals.

In [4]:
root = Path(DATA_ROOT)
ramsey_folders = sorted(root.glob(RAMSEY_FOLDER_PATTERN))
ramsey_folders = [
    d for d in ramsey_folders
    if d.is_dir() and (d / "ds_raw.h5").exists() and (d / "node.json").exists()
]
print(f"Found {len(ramsey_folders)} charge_gate_ramsey folders")

Found 203 charge_gate_ramsey folders


In [5]:
analyzer = ChargeGateRamseyAnalyzer()
ramsey_records = []

for sf in ramsey_folders:
    h5_path = sf / "ds_raw.h5"
    node_path = sf / "node.json"

    with open(node_path, "r", encoding="utf-8") as f:
        node = json.load(f)
    run_start = parse_timestamp(node["metadata"]["run_start"])
    run_end = parse_timestamp(node["metadata"]["run_end"])

    try:
        dataset = load_xarray_h5(str(h5_path))
        sq_data = repetition_data(dataset)[0]
        if RAMSEY_SIGNAL_VAR != "signal" and RAMSEY_SIGNAL_VAR in sq_data:
            sq_data = sq_data.rename({RAMSEY_SIGNAL_VAR: "signal"})
    except Exception as e:
        print(f"  [SKIP] {sf.name}: load failed — {e}")
        continue

    try:
        results, figs = analyzer.analyze(sq_data, **ANALYSIS_KWARGS)
        for fig in figs.values():
            plt.close(fig)
    except Exception as e:
        print(f"  [SKIP] {sf.name}: analysis failed — {e}")
        continue

    abscos = results.get("abscos_params") or {}
    ramsey_records.append({
        "run_start": np.datetime64(run_start),
        "run_end": np.datetime64(run_end),
        "abscos_phase": abscos.get("phase", np.nan),
        "abscos_frequency": abscos.get("frequency", np.nan),
        "abscos_success": abscos.get("success", False),
    })

print(f"\nCompleted: {len(ramsey_records)} / {len(ramsey_folders)} Ramsey runs")


Completed: 203 / 203 Ramsey runs


In [6]:
# ---- Build stable intervals ----
ramsey_times = np.array([r["run_start"] for r in ramsey_records])
phases = np.array([r["abscos_phase"] for r in ramsey_records])
freqs = np.array([r["abscos_frequency"] for r in ramsey_records])

freq_mid = np.nanmean(freqs)
half_period = 1.0 / (2.0 * freq_mid)


def wrapped_phase_delta(phi_a, phi_b, half_period):
    """Shortest-arc distance on the |cos| phase circle."""
    raw = abs(phi_b - phi_a) % half_period
    return min(raw, half_period - raw)


stable_intervals = []
for i in range(len(ramsey_times) - 1):
    delta = wrapped_phase_delta(phases[i], phases[i + 1], half_period)
    if delta < PHASE_THRESHOLD:
        stable_intervals.append((ramsey_times[i], ramsey_times[i + 1]))

print(f"Found {len(stable_intervals)} stable intervals (threshold={PHASE_THRESHOLD})")
for idx, (t0, t1) in enumerate(stable_intervals):
    print(f"  [{idx}] {str(t0)[:19]} → {str(t1)[:19]}")

Found 93 stable intervals (threshold=0.01)
  [0] 2026-04-14T22:37:32 → 2026-04-14T22:37:59
  [1] 2026-04-14T22:37:59 → 2026-04-14T22:38:27
  [2] 2026-04-14T22:38:27 → 2026-04-14T22:38:55
  [3] 2026-04-14T22:39:21 → 2026-04-14T22:39:46
  [4] 2026-04-14T22:39:46 → 2026-04-14T22:40:12
  [5] 2026-04-14T22:41:05 → 2026-04-14T22:41:31
  [6] 2026-04-14T22:41:54 → 2026-04-14T22:42:19
  [7] 2026-04-14T22:42:42 → 2026-04-14T22:43:06
  [8] 2026-04-14T22:43:06 → 2026-04-14T22:43:32
  [9] 2026-04-14T22:43:32 → 2026-04-14T22:43:58
  [10] 2026-04-14T22:43:58 → 2026-04-14T22:44:25
  [11] 2026-04-14T22:44:25 → 2026-04-14T22:44:49
  [12] 2026-04-14T22:44:49 → 2026-04-14T22:45:14
  [13] 2026-04-14T22:46:01 → 2026-04-14T22:46:26
  [14] 2026-04-14T22:46:26 → 2026-04-14T22:46:49
  [15] 2026-04-14T22:46:49 → 2026-04-14T22:47:16
  [16] 2026-04-14T22:47:16 → 2026-04-14T22:47:42
  [17] 2026-04-14T22:47:42 → 2026-04-14T22:48:07
  [18] 2026-04-14T22:48:07 → 2026-04-14T22:48:33
  [19] 2026-04-14T22:48:57 → 2026-04

## 3. Discover & Parse Readout-Power Folders

In [7]:
ro_subfolders = sorted(root.glob(RO_FOLDER_PATTERN))
ro_subfolders = [
    d for d in ro_subfolders
    if d.is_dir() and (d / "ds_raw.h5").exists() and (d / "node.json").exists()
]
print(f"Found {len(ro_subfolders)} readout-power folders")

# Parse folder names → (pulse_name, repeat_index, path, run_start)
ro_entries = []
for sf in ro_subfolders:
    m = re.search(RO_NAME_REGEX, sf.name)
    if m is None:
        print(f"  [SKIP] cannot parse: {sf.name}")
        continue

    repeat_idx = int(m.group(1))
    pulse_name = m.group(2)

    # Read run_start from node.json
    with open(sf / "node.json", "r", encoding="utf-8") as f:
        node = json.load(f)
    run_start = parse_timestamp(node["metadata"]["run_start"])

    ro_entries.append({
        "folder": sf,
        "pulse_name": pulse_name,
        "repeat_idx": repeat_idx,
        "run_start": np.datetime64(run_start),
    })

# Group by pulse name
pulse_groups = defaultdict(list)
for entry in ro_entries:
    pulse_groups[entry["pulse_name"]].append(entry)

print(f"\nPulse names found ({len(pulse_groups)}):")
for pname, entries in sorted(pulse_groups.items()):
    repeats = sorted(set(e["repeat_idx"] for e in entries))
    print(f"  {pname:40s}  repeats={repeats}  total={len(entries)} folders")

Found 202 readout-power folders

Pulse names found (5):
  readout_square                            repeats=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39]  total=40 folders
  readout_three_step                        repeats=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40]  total=41 folders
  readout_three_step_1000                   repeats=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39]  total=40 folders
  readout_three_step_200                    repeats=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40]  total=41 folders
  readout_three_step_400                    repeats=

## 4. Filter by Stable Intervals & Merge per Pulse

In [8]:
def is_in_stable_interval(t, intervals):
    """Check whether timestamp t falls inside any stable interval."""
    for t_lo, t_hi in intervals:
        if t_lo <= t <= t_hi:
            return True
    return False


pulse_datasets = {}  # pulse_name -> merged xr.Dataset

for pulse_name, entries in sorted(pulse_groups.items()):
    print(f"\n{'='*60}")
    print(f"Pulse: {pulse_name}  ({len(entries)} folders)")

    datasets = []
    run_starts = []

    for entry in entries:
        sf = entry["folder"]
        t = entry["run_start"]

        # Stability filter
        if not is_in_stable_interval(t, stable_intervals):
            print(f"  [UNSTABLE] {sf.name}")
            continue

        # Load data
        try:
            dataset = load_xarray_h5(str(sf / "ds_raw.h5"))
            sq_data = repetition_data(dataset)[0]
        except Exception as e:
            print(f"  [SKIP] {sf.name}: load failed — {e}")
            continue

        missing = [v for v in RO_SIGNAL_VARS if v not in sq_data]
        if missing:
            print(f"  [SKIP] {sf.name}: missing vars {missing}")
            continue

        datasets.append(sq_data[RO_SIGNAL_VARS])
        run_starts.append(t)
        print(f"  [OK]       {sf.name}  repeat={entry['repeat_idx']}")

    if len(datasets) == 0:
        print(f"  => No stable runs for pulse '{pulse_name}', skipping.")
        continue

    # Concatenate along run_start
    for i, ds in enumerate(datasets):
        datasets[i] = ds.expand_dims(run_start=[run_starts[i]])
    combined = xr.concat(datasets, dim="run_start")
    combined.attrs["pulse_name"] = pulse_name
    combined.attrs["abscos_frequency_fixed"] = ANALYSIS_KWARGS.get("abscos_frequency_fixed", "")

    pulse_datasets[pulse_name] = combined
    print(f"  => Merged {len(datasets)} runs  dims={dict(combined.sizes)}")

print(f"\n{'='*60}")
print(f"Total pulse groups with data: {len(pulse_datasets)}")


Pulse: readout_square  (40 folders)
  [OK]       #2306_LCH_charge_gate_readout_power_with_ref_0_readout_square_223946  repeat=0
  [UNSTABLE] #2316_LCH_charge_gate_readout_power_with_ref_1_readout_square_224154
  [OK]       #2326_LCH_charge_gate_readout_power_with_ref_2_readout_square_224358  repeat=2
  [UNSTABLE] #2336_LCH_charge_gate_readout_power_with_ref_3_readout_square_224601
  [OK]       #2346_LCH_charge_gate_readout_power_with_ref_4_readout_square_224807  repeat=4
  [OK]       #2356_LCH_charge_gate_readout_power_with_ref_5_readout_square_225009  repeat=5
  [UNSTABLE] #2366_LCH_charge_gate_readout_power_with_ref_6_readout_square_225217
  [OK]       #2376_LCH_charge_gate_readout_power_with_ref_7_readout_square_225417  repeat=7
  [OK]       #2386_LCH_charge_gate_readout_power_with_ref_8_readout_square_225625  repeat=8
  [UNSTABLE] #2396_LCH_charge_gate_readout_power_with_ref_9_readout_square_225835
  [UNSTABLE] #2406_LCH_charge_gate_readout_power_with_ref_10_readout_square_230039


## 5. Save One HDF5 per Pulse

In [9]:
saved_files = {}

for pulse_name, ds in pulse_datasets.items():
    out_path = os.path.join(OUTPUT_DIR, f"ro_{pulse_name}.h5")

    gc.collect()
    if os.path.exists(out_path):
        os.remove(out_path)

    ds.to_netcdf(out_path, engine="h5netcdf", mode="w")
    saved_files[pulse_name] = out_path
    print(f"  Saved: {out_path}")

print(f"\nDone — {len(saved_files)} files saved to {OUTPUT_DIR}")

  Saved: D:\SynologyDrive\LiChiehHsiao\AS\SynologyDrive\data\MIST\20260402\Q1\ng01_ref0165_test04\MWF\op_A13_ngv08_5T\per_pulse\ro_readout_square.h5
  Saved: D:\SynologyDrive\LiChiehHsiao\AS\SynologyDrive\data\MIST\20260402\Q1\ng01_ref0165_test04\MWF\op_A13_ngv08_5T\per_pulse\ro_readout_three_step.h5
  Saved: D:\SynologyDrive\LiChiehHsiao\AS\SynologyDrive\data\MIST\20260402\Q1\ng01_ref0165_test04\MWF\op_A13_ngv08_5T\per_pulse\ro_readout_three_step_1000.h5
  Saved: D:\SynologyDrive\LiChiehHsiao\AS\SynologyDrive\data\MIST\20260402\Q1\ng01_ref0165_test04\MWF\op_A13_ngv08_5T\per_pulse\ro_readout_three_step_200.h5
  Saved: D:\SynologyDrive\LiChiehHsiao\AS\SynologyDrive\data\MIST\20260402\Q1\ng01_ref0165_test04\MWF\op_A13_ngv08_5T\per_pulse\ro_readout_three_step_400.h5

Done — 5 files saved to D:\SynologyDrive\LiChiehHsiao\AS\SynologyDrive\data\MIST\20260402\Q1\ng01_ref0165_test04\MWF\op_A13_ngv08_5T\per_pulse


## 6. Also Save Shot-Merged Version (optional)
Stack `shot_idx` across all `run_start` values into a single long dimension per pulse.

In [10]:
for pulse_name, ds in pulse_datasets.items():
    merged = (
        ds[RO_SIGNAL_VARS]
        .stack(merged_shot=("run_start", "shot_idx"))
        .reset_index("merged_shot", drop=True)
        .rename({"merged_shot": "shot_idx"})
    )
    merged.attrs.update(ds.attrs)

    out_path = os.path.join(OUTPUT_DIR, f"ro_{pulse_name}_merged.h5")
    gc.collect()
    if os.path.exists(out_path):
        os.remove(out_path)
    merged.to_netcdf(out_path, engine="h5netcdf", mode="w")
    print(f"  Saved merged: {out_path}")

  Saved merged: D:\SynologyDrive\LiChiehHsiao\AS\SynologyDrive\data\MIST\20260402\Q1\ng01_ref0165_test04\MWF\op_A13_ngv08_5T\per_pulse\ro_readout_square_merged.h5
  Saved merged: D:\SynologyDrive\LiChiehHsiao\AS\SynologyDrive\data\MIST\20260402\Q1\ng01_ref0165_test04\MWF\op_A13_ngv08_5T\per_pulse\ro_readout_three_step_merged.h5
  Saved merged: D:\SynologyDrive\LiChiehHsiao\AS\SynologyDrive\data\MIST\20260402\Q1\ng01_ref0165_test04\MWF\op_A13_ngv08_5T\per_pulse\ro_readout_three_step_1000_merged.h5
  Saved merged: D:\SynologyDrive\LiChiehHsiao\AS\SynologyDrive\data\MIST\20260402\Q1\ng01_ref0165_test04\MWF\op_A13_ngv08_5T\per_pulse\ro_readout_three_step_200_merged.h5
  Saved merged: D:\SynologyDrive\LiChiehHsiao\AS\SynologyDrive\data\MIST\20260402\Q1\ng01_ref0165_test04\MWF\op_A13_ngv08_5T\per_pulse\ro_readout_three_step_400_merged.h5


## 7. Summary — Shot Count per Pulse

In [11]:
print(f"{'Pulse name':40s}  {'runs':>5s}  {'shot_idx':>10s}  {'total shots':>12s}")
print("-" * 75)
for pulse_name, ds in sorted(pulse_datasets.items()):
    n_runs = ds.sizes["run_start"]
    n_shots_per_run = ds.sizes["shot_idx"]
    total = n_runs * n_shots_per_run
    print(f"{pulse_name:40s}  {n_runs:5d}  {n_shots_per_run:10d}  {total:12d}")

Pulse name                                 runs    shot_idx   total shots
---------------------------------------------------------------------------
readout_square                               16        4000         64000
readout_three_step                           21        4000         84000
readout_three_step_1000                      18        4000         72000
readout_three_step_200                       17        4000         68000
readout_three_step_400                       21        4000         84000
